In [ ]:
import os
import openai
import sys
sys.path.append('../..')
import shutil  #ensures old data base is cleared
if os.path.exists('./docs/chroma'):
    shutil.rmtree('./docs/chroma')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']



from langchain.document_loaders import PyPDFLoader

pdf_files = [
    "MIL-STD-882E.pdf",
    "DOD SysEng Guidebook.pdf",
    "Scrum-Guide-US-2020.pdf",
    "DOD_DevSecOps Fundamentals.pdf"
]

pages = []
for pdf in pdf_files:
    try:
        loader = PyPDFLoader(pdf)
        pages.extend(loader.load())
        print(f"✅ Loaded: {pdf}")
    except Exception as e:
        print(f"❌ Failed: {pdf} — {e}")

print(f"\nTotal pages loaded: {len(pages)}")



from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)



docs = text_splitter.split_documents(pages)



print(len(pages)) 



print(len(docs))



from langchain.embeddings.openai import OpenAIEmbeddings
embedding = OpenAIEmbeddings()


from langchain_chroma import Chroma



persist_directory = 'docs/chroma/'


vectordb = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory=persist_directory
)



print(vectordb._collection.count())


from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)



from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# We are DEFINING the tool here
def get_document_answer(user_query, pipeline):
    """
    Executes a query using the modern .invoke() method and handles history.
    """
    # We pass BOTH the question and an empty list for chat_history
    # Using .invoke() instead of () fixes the Deprecation Warnings
    result = pipeline.invoke({
        "question": user_query, 
        "chat_history": [] 
    })
    
    print(f"ANALYST RESPONSE:\n{result['answer']}\n")
    
    print("CITATIONS:")
    seen_sources = set()
    for doc in result["source_documents"]:
        source_name = doc.metadata.get('source', 'Unknown Document')
        page_num = doc.metadata.get('page', 'N/A')
        citation = f"Ref: {source_name} (Page {page_num})"
        
        if citation not in seen_sources:
            print(f"  - {citation}")
            seen_sources.add(citation)
    print("-" * 40)       

from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever

# Define the 'Rules of the Road' for the AI
metadata_field_info = [
    AttributeInfo(
        name="source",
        description="The name of the PDF document (e.g., MIL-STD-882E.pdf)",
        type="string",
    ),
    AttributeInfo(
        name="page",
        description="The specific page number in the document",
        type="integer",
    ),
]

document_content_description = "Technical guidebooks and safety standards"

# This creates a 'Smart Librarian' that filters BEFORE searching
self_query_retriever = SelfQueryRetriever.from_llm(
    llm,
    vectordb,
    document_content_description,
    metadata_field_info,
    verbose=True # Set this to True to see the 'Filter' in action
)

# 1. Initialize the ranker WITH the model name (Crucial for 2026)
from flashrank import Ranker
ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

# 2. This is the correct, updated path for 2026
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain.retrievers import ContextualCompressionRetriever

# 3. Grab 10 'candidates' from the database
base_retriever = vectordb.as_retriever(search_kwargs={"k": 10})

# 4. Use a Re-ranker to sort them by actual relevance
compressor = FlashrankRerank(model="ms-marco-MiniLM-L-12-v2", top_n=3)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=base_retriever
)

from langchain.chains import ConversationalRetrievalChain

chat_history = []

qa_chain = ConversationalRetrievalChain.from_llm(
    llm,
    retriever=compression_retriever,
    return_source_documents=True,
)
   
get_document_answer("What does the Scrum guide say?", qa_chain)
get_document_answer("tell me about Agile?", qa_chain)
get_document_answer("what are the consequences of poor requirements?", qa_chain)

